In [1]:
!pip install yfinance

In [1]:
"""
Interactive Brokers Paper Trading Algorithm
Monitors price changes and trades based on momentum shifts
"""

import yfinance as yf
import time
from ib_insync import IB, Stock, MarketOrder, StopOrder, util
from collections import deque
import asyncio
import nest_asyncio

# Allow nested event loops (fixes "event loop already running" error)
nest_asyncio.apply()

# Configuration
SYMBOLS = ["MSFT"]  # US stocks - add your desired symbols
# For UK stocks, use format: "TICKER.L" (e.g., "LLOY.L", "BP.L", "BARC.L")
# SYMBOLS = ["LLOY.L", "BP.L", "BARC.L"]  # Example UK stocks

EXCHANGE = "US"  # "US" or "UK"
NEUTRAL_THRESHOLD = 0.0001  # Price change threshold for "neutral" (0.01%)
CHECK_INTERVAL = 1.0  # Check every 1 second (yfinance updates every 1-5 seconds)
STOP_LOSS_PERCENT = 0.01  # Stop loss of 1% per share (works for any price)
TOTAL_BUDGET = 10000.0  # Total budget in USD or GBP
BUDGET_PER_SYMBOL = None  # None = equal split, or set specific amount per symbol

class MomentumTradingAlgo:
    def __init__(self, symbols, exchange, neutral_threshold, stop_loss_percent, budget):
        self.symbols = symbols if isinstance(symbols, list) else [symbols]
        self.exchange = exchange.upper()
        self.neutral_threshold = neutral_threshold
        self.stop_loss_percent = stop_loss_percent
        self.total_budget = budget
        self.available_cash = budget
        self.ib = IB()
        
        # Track data per symbol
        self.price_history = {symbol: deque(maxlen=2) for symbol in self.symbols}
        self.change_history = {symbol: deque(maxlen=2) for symbol in self.symbols}
        self.open_positions = {symbol: {} for symbol in self.symbols}  # {symbol: {trade_id: {...}}}
        
        # Set currency and exchange parameters
        self.currency = "GBP" if self.exchange == "UK" else "USD"
        self.ib_exchange = "LSE" if self.exchange == "UK" else "SMART"
        
    def connect_ib(self):
        """Connect to Interactive Brokers TWS or IB Gateway"""
        try:
            # Connect to IB Gateway (paper trading typically on port 4002)
            # For TWS paper trading, use port 7497
            # For TWS live trading, use port 7496
            # For IB Gateway live trading, use port 4001
            
            self.ib.connect('192.168.1.199', 7497, clientId=328)
            print("✓ Connected to Interactive Brokers (Paper Trading)")
            return True
        except Exception as e:
            print(f"✗ Error connecting to IB: {e}")
            print("\nTroubleshooting steps:")
            print("1. Make sure IB Gateway or TWS is running")
            print("2. Check you're using the correct port:")
            print("   - IB Gateway Paper Trading: 4002")
            print("   - TWS Paper Trading: 7497")
            print("   - IB Gateway Live Trading: 4001")
            print("   - TWS Live Trading: 7496")
            print("3. Enable API connections in TWS/Gateway settings:")
            print("   - Go to: Edit → Global Configuration → API → Settings")
            print("   - Check 'Enable ActiveX and Socket Clients'")
            print("   - Uncheck 'Read-Only API'")
            print("4. Add clientId=1 to trusted clients in API settings")
            return False
    
    def get_current_price(self, symbol):
        """Get current price using yfinance"""
        try:
            ticker = yf.Ticker(symbol)
            data = ticker.history(period="1d", interval="1m")
            if not data.empty:
                return data['Close'].iloc[-1]
            return None
        except Exception as e:
            print(f"Error fetching price for {symbol}: {e}")
            return None
    
    def calculate_change(self, current_price, previous_price):
        """Calculate percentage change between prices"""
        if previous_price is None or previous_price == 0:
            return 0
        return (current_price - previous_price) / previous_price
    
    def classify_change(self, change):
        """Classify price change as positive, negative, or neutral"""
        if abs(change) <= self.neutral_threshold:
            return "neutral"
        elif change > 0:
            return "positive"
        else:
            return "negative"
    
    def get_ib_symbol(self, symbol):
        """Convert yfinance symbol to IB format"""
        # For UK stocks, remove the .L suffix for IB
        if self.exchange == "UK" and symbol.endswith(".L"):
            return symbol[:-2]
        return symbol
    
    def place_order(self, symbol, action):
        """Place buy or sell order with Interactive Brokers and set stop loss"""
        try:
            # Calculate position size based on available cash
            current_price = self.get_current_price(symbol)
            if current_price is None:
                print(f"Cannot place order for {symbol}: Unable to get current price")
                return None
            
            if action == "BUY":
                # Calculate max shares we can buy
                max_shares = int(self.available_cash / current_price)
                if max_shares <= 0:
                    print(f"Insufficient funds for {symbol}. Available: {self.currency}{self.available_cash:.2f}, Price: {self.currency}{current_price:.2f}")
                    return None
                quantity = max_shares
            else:  # SELL
                # Calculate total position size for this symbol
                total_position = sum(pos['quantity'] for pos in self.open_positions[symbol].values())
                if total_position <= 0:
                    print(f"No open positions to sell for {symbol}")
                    return None
                quantity = total_position
            
            # Create contract with proper symbol format
            ib_symbol = self.get_ib_symbol(symbol)
            contract = Stock(ib_symbol, self.ib_exchange, self.currency)
            self.ib.qualifyContracts(contract)
            
            # Place market order
            order = MarketOrder(action, quantity)
            trade = self.ib.placeOrder(contract, order)
            
            print(f"\n{'='*50}")
            print(f"Order placed: {action} {quantity} shares of {symbol}")
            print(f"Entry price: {self.currency}{current_price:.2f}")
            
            # Wait for order to fill
            self.ib.sleep(2)
            
            if action == "BUY":
                # Update available cash
                cost = quantity * current_price
                self.available_cash -= cost
                
                # Place stop loss order (percentage-based)
                stop_price = current_price * (1 - self.stop_loss_percent)
                stop_order = StopOrder("SELL", quantity, stop_price)
                stop_trade = self.ib.placeOrder(contract, stop_order)
                
                # Store position info
                self.open_positions[symbol][trade.order.orderId] = {
                    'entry_price': current_price,
                    'quantity': quantity,
                    'stop_order': stop_trade
                }
                
                print(f"Stop loss set at: {self.currency}{stop_price:.2f} ({self.stop_loss_percent*100}% below entry)")
                print(f"Available cash: {self.currency}{self.available_cash:.2f}")
            else:  # SELL
                # Cancel all stop orders and clear positions for this symbol
                for pos in self.open_positions[symbol].values():
                    if pos['stop_order']:
                        self.ib.cancelOrder(pos['stop_order'].order)
                
                # Update available cash
                proceeds = quantity * current_price
                self.available_cash += proceeds
                
                # Clear positions for this symbol
                self.open_positions[symbol].clear()
                
                print(f"All positions closed for {symbol}")
                print(f"Available cash: {self.currency}{self.available_cash:.2f}")
            
            print(f"{'='*50}\n")
            return trade
            
        except Exception as e:
            print(f"Error placing order for {symbol}: {e}")
            return None
    
    def run(self):
        """Main trading loop"""
        if not self.connect_ib():
            return
        
        print(f"Starting momentum trading algorithm")
        print(f"Exchange: {self.exchange}")
        print(f"Currency: {self.currency}")
        print(f"Symbols: {', '.join(self.symbols)}")
        print(f"Neutral threshold: {self.neutral_threshold * 100}%")
        print(f"Check interval: {CHECK_INTERVAL} seconds")
        print(f"Stop loss: {self.stop_loss_percent*100}% below entry")
        print(f"Total budget: {self.currency}{self.total_budget:.2f}")
        print(f"Available cash: {self.currency}{self.available_cash:.2f}\n")
        
        try:
            while True:
                # Process each symbol
                for symbol in self.symbols:
                    # Get current price
                    current_price = self.get_current_price(symbol)
                    
                    if current_price is None:
                        continue
                    
                    # Calculate change if we have previous price
                    if len(self.price_history[symbol]) > 0:
                        previous_price = self.price_history[symbol][-1]
                        current_change = self.calculate_change(current_price, previous_price)
                        current_state = self.classify_change(current_change)
                        
                        # Check for trading signals if we have previous change
                        if len(self.change_history[symbol]) > 0:
                            previous_change = self.change_history[symbol][-1]
                            previous_state = self.classify_change(previous_change)
                            
                            print(f"{symbol} | Price: {self.currency}{current_price:.2f} | Change: {current_change*100:.4f}% | State: {current_state}")
                            
                            # Trading logic
                            # If previous state was neutral and current is negative -> BUY
                            if previous_state == "neutral" and current_state == "negative":
                                print(f"SIGNAL [{symbol}]: Neutral -> Negative: BUY")
                                self.place_order(symbol, "BUY")
                            
                            # If previous state was neutral and current is positive -> SELL
                            elif previous_state == "neutral" and current_state == "positive":
                                print(f"SIGNAL [{symbol}]: Neutral -> Positive: SELL")
                                self.place_order(symbol, "SELL")
                        
                        # Store current change
                        self.change_history[symbol].append(current_change)
                    
                    # Store current price
                    self.price_history[symbol].append(current_price)
                
                # Wait for next check
                time.sleep(CHECK_INTERVAL)
                
        except KeyboardInterrupt:
            print("\nStopping algorithm...")
        finally:
            self.ib.disconnect()
            print("Disconnected from Interactive Brokers")

if __name__ == "__main__":
    # Initialize and run the algorithm
    algo = MomentumTradingAlgo(
        symbols=SYMBOLS,
        exchange=EXCHANGE,
        neutral_threshold=NEUTRAL_THRESHOLD,
        stop_loss_percent=STOP_LOSS_PERCENT,
        budget=TOTAL_BUDGET
    )
    algo.run()

KeyboardInterrupt: 

API connection failed: TimeoutError()
